# AdaptiveHb - Transparent End-to-End Pipeline (with proof of training)

One notebook. It takes your input **exactly** as `train_pipeline.ipynb` does, then
walks every stage of the pipeline **out loud** so you can see what actually runs:

1. **Environment truth** - is PyTorch here? are the real backbones registered, or is
   the framework silently falling back to constant *reference* models?
2. **Dataset load & validate** - how many patients / images / masks, Hb range.
3. **Split** - patient-level train/val/test sizes, per tissue, disjointness check.
4. **Preprocessing agent** - proves the agent runs: kept/dropped samples, Hb gate,
   balanced-bin weights.
5. **Segmentation training** - which model(s), real vs reference, params, per-epoch
   loss, and a **weight-norm before/after** check that proves gradients moved weights.
6. **Prediction training** - same, per tissue.
7. **Agents** - every enabled agent and the decision it makes.
8. **Evaluation** - MAE/RMSE/R2, plus *prediction provenance*: did each tissue model
   load a trained checkpoint, and what does it actually output on a test batch?
9. **Verdict** - a one-screen summary answering 'is it training, and are the models used?'

> Nothing here re-implements the pipeline. Every stage calls the **same framework
> functions** the real run uses (`adaptivehb.managers.pipeline_modes`, the managers,
> `TrainingManager`) - only wrapped in printing and verification.

## 0 - Locate the repo and make the framework importable

In [ ]:
import sys, os, json, math, textwrap
from pathlib import Path

def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for directory in (start, *start.parents):
        if (directory / 'configs' / 'project.yaml').is_file():
            return directory
    raise FileNotFoundError('Run this notebook from inside the AgenticHb repository.')

REPO_ROOT = find_repo_root()
SRC = REPO_ROOT / 'src'
if SRC.is_dir() and str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
CONFIG_DIR = REPO_ROOT / 'configs'
print('Repository root :', REPO_ROOT)
print('Config directory:', CONFIG_DIR)

## 1 - USER SETTINGS - the ONLY cell you edit

Identical inputs to `train_pipeline.ipynb`. Point the paths at your real dataset.
`USE_SYNTHETIC = True` fabricates a tiny dataset so you can dry-run the *machinery*
anywhere; set it to `False` (default) to run on your real EDA data.

In [ ]:
# --- DRY RUN ----------------------------------------------------
USE_SYNTHETIC = False     # False => use the real paths below; True => fabricate a tiny dataset

# ================================================================
# 1) SEGMENTATION - enable per tissue, give its image + mask dirs.
# ================================================================
EYES_SEG          = True
EYE_SEG_IMAGES    = 'Segmentation/EYE SEG/original'
EYE_SEG_MASKS     = 'Segmentation/EYE SEG/annotations'

NAIL_SEG          = True
NAIL_SEG_IMAGES   = 'Segmentation/NAIL SEG/ALLNAILSWITHMASKS'
NAIL_SEG_MASKS    = 'Segmentation/NAIL SEG/ALLMASKS'

TONGUE_SEG        = True
TONGUE_SEG_IMAGES = 'Segmentation/TONGUE SEG/JPEGimages'
TONGUE_SEG_MASKS  = 'Segmentation/TONGUE SEG/SegmentationClassBW'

PALM_SEG          = True
PALM_SEG_IMAGES   = 'Segmentation/PALM SEG/JPEGimages'
PALM_SEG_MASKS    = 'Segmentation/PALM SEG/SegmentationClassBW'

SEG_RUN_ON = ['eye', 'palm', 'nail', 'tongue']   # tissues to train segmentation on

# ================================================================
# 2) PREDICTION - one image path PER SIDE.
# ================================================================
LEFT_EYE   = 'dataset/left_eye'
RIGHT_EYE  = 'dataset/right_eye'
LEFT_PALM  = 'dataset/palm_left'
RIGHT_PALM = 'dataset/palm_right'
LEFT_NAIL  = 'dataset/left_nail'
RIGHT_NAIL = 'dataset/right_nail'
TONGUE     = 'dataset/tongue'

PRED_RUN_ON = ['eye', 'palm', 'nail', 'tongue']   # tissues to train prediction on
SAMPLING_MODE = 'extended'   # 'extended' => each image is its own data point

# ================================================================
# 3) LABELS - ONE Excel/CSV file + the column names you need.
# ================================================================
METADATA_FILE     = 'dataset/merge_excel_1.csv'
DATASET_ROOT      = None      # None => METADATA_FILE's folder
PATIENT_ID_COLUMN = 'Patient ID'
TARGET_COLUMN     = 'Haemoglobin (gm/dL)'
HEIGHT_COLUMN     = 'Height'
WEIGHT_COLUMN     = 'Weight'
BMI_COLUMN        = 'BMI'
METADATA_COLUMNS  = ['patient_id', 'hemoglobin', 'height', 'weight', 'BMI']

# ================================================================
# 4) RUN OUTPUT + MODELS
# ================================================================
BASE_DIR        = None            # None => <repo>/runs
EXPERIMENT_NAME = 'transparent_run'
EPOCHS          = 3               # keep small while verifying the flow; raise later

SEG_MODELS         = None         # None keeps the config default
PRED_BACKBONES     = None
PRED_DEFAULT       = None
PRED_TISSUE_MODELS = None

print('User settings loaded - edit only this cell.')

## 2 - Environment truth - *is real training even possible here?*

This is the first thing to check when the MAE looks wrong. If PyTorch is missing,
**every** backbone silently falls back to a torch-free *reference* model that returns
a constant Hb - no learning happens at all. The banner below tells you unambiguously
which mode you are in.

In [ ]:
def banner(title, ok):
    mark = 'OK ' if ok else 'XX '
    print('=' * 64)
    print(f'{mark} {title}')
    print('=' * 64)

try:
    import torch, torchvision
    TORCH_OK = True
except Exception as e:
    torch = None; TORCH_OK = False; _torch_err = e

try:
    import segmentation_models_pytorch as smp
    SMP_OK = True; smp_ver = smp.__version__
except Exception:
    SMP_OK = False; smp_ver = None

from adaptivehb.segmentation.registry import available_segmentation
from adaptivehb.prediction.registry import available_prediction
seg_avail  = available_segmentation()
pred_avail = available_prediction()

print('PyTorch installed          :', TORCH_OK, ('(' + torch.__version__ + ')') if TORCH_OK else '')
if TORCH_OK:
    print('CUDA available             :', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('CUDA device                :', torch.cuda.get_device_name(0))
print('segmentation_models_pytorch:', SMP_OK, ('(' + str(smp_ver) + ')') if SMP_OK else '(needed for segformer)')
print('Registered SEG builders    :', seg_avail)
print('Registered PRED builders   :', pred_avail)
print()

REAL_PRED = any(b in pred_avail for b in ('efficientnet','resnet','densenet','vit','convnext'))
REAL_SEG  = any(b in seg_avail  for b in ('unet','deeplabv3plus','segformer'))
if REAL_PRED:
    banner('REAL prediction backbones are registered - training will learn.', True)
else:
    banner('NO real prediction backbones - falling back to CONSTANT reference model. '
           'Install torch:  pip install -e ".[ml]"', False)
if not REAL_SEG:
    banner('NO real segmentation backbones - segmentation will use the reference model.', False)
if os.environ.get('ADAPTIVEHB_FORCE_REFERENCE'):
    banner('ADAPTIVEHB_FORCE_REFERENCE is set - everything is forced to reference. Unset it to train.', False)

## 3 - Apply settings to the config (files on disk untouched)

Same overlay logic as your training notebook: builds `tissue_sources`, wires masks,
sets sampling mode, points the metadata loader at your file. BMI is derived on load.

In [ ]:
from adaptivehb.config import ConfigLoader
from adaptivehb.dataset import generate_synthetic_dataset

if BASE_DIR is None:
    BASE_DIR = str(REPO_ROOT / 'runs')
IMAGES_SUBDIR, MASKS_SUBDIR = 'images', 'masks'

_PRED_SIDES = {
    'eye':    {'left': LEFT_EYE,  'right': RIGHT_EYE},
    'palm':   {'left': LEFT_PALM, 'right': RIGHT_PALM},
    'nail':   {'left': LEFT_NAIL, 'right': RIGHT_NAIL},
    'tongue': {'center': TONGUE},
}
_SEG = {
    'eye':    {'on': EYES_SEG   and 'eye'    in SEG_RUN_ON, 'images': EYE_SEG_IMAGES,    'masks': EYE_SEG_MASKS},
    'nail':   {'on': NAIL_SEG   and 'nail'   in SEG_RUN_ON, 'images': NAIL_SEG_IMAGES,   'masks': NAIL_SEG_MASKS},
    'tongue': {'on': TONGUE_SEG and 'tongue' in SEG_RUN_ON, 'images': TONGUE_SEG_IMAGES, 'masks': TONGUE_SEG_MASKS},
    'palm':   {'on': PALM_SEG   and 'palm'   in SEG_RUN_ON, 'images': PALM_SEG_IMAGES,   'masks': PALM_SEG_MASKS},
}

def _build_tissue_sources():
    sources, tissues = {}, []
    considered = set(PRED_RUN_ON) | {t for t, s in _SEG.items() if s['on']}
    for tissue, side_paths in _PRED_SIDES.items():
        if tissue not in considered:
            continue
        seg = _SEG.get(tissue, {})
        seg_masks = seg.get('masks') if seg.get('on') else None
        sides = {}
        for side, images in side_paths.items():
            if images:
                entry = {'images': images}
                if seg_masks:
                    entry['masks'] = seg_masks
                sides[side] = entry
        if sides:
            sources[tissue] = {'sides': sides}; tissues.append(tissue)
        elif seg.get('on') and seg.get('images'):
            sources[tissue] = {'images': seg['images'], 'masks': seg.get('masks')}; tissues.append(tissue)
    return sources, tissues

config = ConfigLoader(CONFIG_DIR).load()
ds   = config.section('dataset')['dataset']
seg  = config.section('segmentation')['segmentation']
pred = config.section('prediction')['prediction']

if USE_SYNTHETIC:
    DATASET_ROOT = str(Path(BASE_DIR) / 'synthetic_dataset')
    syn_tissues = list(dict.fromkeys(list(PRED_RUN_ON) + list(SEG_RUN_ON))) or ['eye']
    generate_synthetic_dataset(DATASET_ROOT, num_patients=24, seed=7, tissues=syn_tissues)
    ds['images_dir'], ds['masks_dir'] = IMAGES_SUBDIR, MASKS_SUBDIR
    ds['metadata_file'] = 'metadata/patients.csv'
    ds['tissues'] = syn_tissues
    ds['tissue_sources'] = {}
    ds['sampling_mode'] = SAMPLING_MODE
    ds['metadata'] = {**ds.get('metadata', {}),
                      'patient_id_column': 'Patient_ID', 'target_column': 'Hemoglobin',
                      'mandatory_columns': ['Patient_ID', 'Hemoglobin'],
                      'needed_columns': ['Patient_ID', 'Hemoglobin', 'Age', 'Gender', 'Height', 'Weight', 'BMI'],
                      'height_column': 'Height', 'weight_column': 'Weight',
                      'bmi_column': 'BMI', 'compute_bmi': True}
    print('Synthetic dataset at:', DATASET_ROOT, '| tissues:', syn_tissues)
else:
    if DATASET_ROOT is None:
        DATASET_ROOT = str(Path(METADATA_FILE).resolve().parent)
    tissue_sources, active_tissues = _build_tissue_sources()
    ds['metadata_file'] = METADATA_FILE
    ds['images_dir'], ds['masks_dir'] = IMAGES_SUBDIR, MASKS_SUBDIR
    ds['tissues'] = active_tissues
    ds['tissue_sources'] = tissue_sources
    ds['sampling_mode'] = SAMPLING_MODE
    ds['metadata'] = {**ds.get('metadata', {}),
                      'patient_id_column': PATIENT_ID_COLUMN, 'target_column': TARGET_COLUMN,
                      'mandatory_columns': [PATIENT_ID_COLUMN, TARGET_COLUMN],
                      'needed_columns': list(METADATA_COLUMNS),
                      'height_column': HEIGHT_COLUMN, 'weight_column': WEIGHT_COLUMN,
                      'bmi_column': BMI_COLUMN, 'compute_bmi': True}
    print('Labels file    :', METADATA_FILE)
    print('Dataset root   :', DATASET_ROOT)
    print('Active tissues :', active_tissues or '(NONE - set at least one prediction side path!)')
    print('Segmentation on:', [t for t, s in _SEG.items() if s['on']] or '(none)')

if SEG_MODELS is not None:
    seg['available_models'] = list(SEG_MODELS); seg['default_model'] = SEG_MODELS[0]
if PRED_BACKBONES is not None:
    pred['available_models'] = list(PRED_BACKBONES)
if PRED_DEFAULT is not None:
    pred['default_model'] = PRED_DEFAULT
if PRED_TISSUE_MODELS is not None:
    pred['tissue_models'] = dict(PRED_TISSUE_MODELS)

print('Output dir     :', BASE_DIR)
print('Segmentation   :', seg['available_models'], '(default:', seg['default_model'] + ')')
print('Prediction     :', pred['available_models'], '(default:', pred['default_model'] + ')')
print('Epochs         :', EPOCHS)

## 4 - Build the pipeline and initialize the managers

`HbPipeline` is the single public facade. `initialize()` seeds the RNGs and boots
every manager in the documented order. We keep the internal `PipelineManager` (`pm`)
so the rest of the notebook can drive each stage explicitly.

In [ ]:
from adaptivehb.pipeline import HbPipeline

pipeline = HbPipeline(config, base_dir=BASE_DIR, dataset_root=DATASET_ROOT)
pipeline.initialize()
pm = pipeline.manager   # internal orchestrator; all managers hang off this

print('Seed              :', pm.config.project.seed)
print('Managers online   :', [n for n in (
    'registry','state','checkpoints','experiments','dataset','segmentation',
    'prediction','agents','evaluation','reporting','deployment','training') if getattr(pm, n, None)])
print('Enabled agents    :', pm.agents.enabled_agents())
print('Trainable factory :', pm.trainable_factory)   # dispatches SEG/PRED -> real; else dummy

## 5 - Stage 1 - Dataset load & validate

Reads your metadata + indexes the images into `Sample` records, then runs the
validator. If Hb is missing or images don't resolve, you see it **here**, before any
model runs.

In [ ]:
samples = pm.dataset.load()
report  = pm.dataset.validate()
stats   = pm.dataset.statistics()

print('Total samples indexed :', len(samples))
print('Patients              :', report.num_patients)
print('Images / Masks        :', report.num_images, '/', report.num_masks)
print('Dataset valid         :', report.is_valid)
if report.errors:
    print('ERRORS  :', [(i.code, i.count) for i in report.errors])
if report.warnings:
    print('WARN    :', [(i.code, i.count) for i in report.warnings][:8])
print('Hb statistics         :', {k: round(v, 3) for k, v in stats.hb.items()} if stats.hb else '(none)')
print('Samples missing Hb    :', stats.missing_hb)
print()
from collections import Counter
by_tissue = Counter(s.tissue for s in samples)
print('Samples per tissue    :', dict(by_tissue))
print('With a mask (seg-able):', sum(1 for s in samples if s.mask_path))
print('Example sample        :', samples[0].to_dict() if samples else '(none)')

## 6 - Stage 2 - Split (patient-level)

Splitting is done on **patients**, not images, so no patient leaks across train/val/test.
The disjointness assertion below fails loudly if that ever breaks.

In [ ]:
split_ids = pm.dataset.split()   # {'train': [...ids], 'validation': [...], 'test': [...]}
print('Split sizes (patients):', {k: len(v) for k, v in split_ids.items()})

for name in ('train', 'validation', 'test'):
    s = pm.dataset.samples(name)
    tset = Counter(x.tissue for x in s)
    hb = [x.hb for x in s if x.hb is not None]
    hb_rng = (round(min(hb),2), round(max(hb),2)) if hb else '(no Hb)'
    print(f'  {name:11s}: {len(s):4d} samples | per-tissue {dict(tset)} | Hb range {hb_rng}')

sets = {n: set(split_ids.get(n, [])) for n in ('train','validation','test')}
leak = (sets['train'] & sets['validation']) | (sets['train'] & sets['test']) | (sets['validation'] & sets['test'])
print()
print('Patient overlap across splits:', leak or 'NONE (no leakage)')

## 7 - Stage 3 - Preprocessing agent (proof it runs)

The `PreprocessingAgent` is the agentic front door to training data. It (a) **gates**
samples to a clinical Hb window and (b) assigns **balanced-bin weights** so the model
can't just predict the mean. We build it the exact way the pipeline does and print its
decision, so you can confirm it fired and see how many samples it kept / dropped.

In [ ]:
from adaptivehb.managers import pipeline_modes as PM

agent = PM._build_preprocessing_agent(pm)
if agent is None:
    print('Preprocessing agent is DISABLED in agents.yaml -> raw samples used as-is.')
else:
    print('Preprocessing agent :', type(agent).__name__, '| enabled =', agent.enabled)
    print('Hb gate             :', agent.spec.hb_filter)
    print('Balanced sampling   :', agent.spec.balanced_sampling)
    print()
    train_samples = list(pm.dataset.samples('train'))
    decision = agent.predict({'samples': train_samples})
    o = decision.outputs
    print('Reason              :', decision.reason)
    print('Kept / Dropped      :', o['kept_samples'], '/', o['dropped_samples'])
    print('Effective Hb range  :', o['hb_range'])
    print('Balanced weights?   :', o['balanced'])
    w = o['sample_weights']
    if w:
        import statistics as st
        print('Weight stats        : min=%.3f max=%.3f mean=%.3f (n=%d)' % (min(w), max(w), st.mean(w), len(w)))
        print('  -> non-uniform weights mean rare Hb samples are drawn more often (anti mean-collapse).')

## 8 - Instrumentation helpers - how we *prove* a model trained

For every model we record: its concrete **class** (real torch vs reference/dummy), the
**parameter count**, and a **weight signature** (sum of parameter L2 norms) *before* and
*after* training. If the signature changes, gradients moved the weights - i.e. it really
learned. A reference/dummy model has no parameters and this is stated explicitly.

In [ ]:
def model_kind(trainable) -> str:
    name = type(trainable).__name__
    if 'Torch' in name:     return 'REAL (torch backbone)'
    if 'Reference' in name: return 'REFERENCE (constant, no learning)'
    if 'Dummy' in name:     return 'DUMMY (no learning)'
    return name

def torch_module(trainable):
    return getattr(trainable, '_module', None)

def param_count(trainable) -> int:
    m = torch_module(trainable)
    if m is None: return 0
    return sum(p.numel() for p in m.parameters())

def weight_signature(trainable):
    '''Sum of L2 norms of all parameters - a scalar fingerprint of the weights.'''
    m = torch_module(trainable)
    if m is None: return None
    import torch as _t
    with _t.no_grad():
        return float(sum(p.detach().float().norm().item() for p in m.parameters()))

def train_one_model(plan, task_label):
    '''Build -> attach data -> train ONE model with full narration. Uses the real
    framework code paths (pm.trainable_factory, PM._attach_dataloaders, pm.training.train).'''
    print('-' * 68)
    print(f'[{task_label}]  plan={plan.name!r}  arch={plan.architecture!r}  epochs={plan.epochs}')
    trainable = pm.trainable_factory(plan)               # same factory the pipeline uses
    kind = model_kind(trainable)
    print('  model class     :', type(trainable).__name__, '->', kind)
    print('  parameters      :', f'{param_count(trainable):,}')
    PM._attach_dataloaders(pm, trainable, plan)          # real dataloaders (+ preprocessing agent)
    before = weight_signature(trainable)
    print('  weight signature (before):', before)
    result = pm.training.train(trainable, plan)          # the real resumable training loop
    after = weight_signature(trainable)
    print('  weight signature (after) :', after)
    if before is not None and after is not None:
        moved = abs(after - before) > 1e-6
        print('  WEIGHTS MOVED?  :', ('YES (real learning)' if moved else 'NO (weights unchanged!)'))
    else:
        print('  WEIGHTS MOVED?  :  n/a - no parameters (reference/dummy, NOT learning)')
    print('  per-epoch history:')
    for h in result.history:
        ep = int(h.get('epoch', 0))
        keys = [k for k in ('train_loss','val_loss','train_mae','val_mae','dice') if k in h]
        print('     epoch %2d | ' % ep + '  '.join(f'{k}={h[k]:.4f}' for k in keys))
    print('  best %s=%.4f @epoch %d | registered_id=%s'
          % (plan.monitor, result.best_metric, result.best_epoch, result.registered_id))
    ck = pm.checkpoints
    saved = ck.exists(plan.name, tag='best') or ck.exists(plan.name, tag='latest')
    print('  checkpoint saved:', 'YES' if saved else 'NO')
    return result

## 9 - Stage 4 - Segmentation models (which run, and do they learn?)

One plan per architecture in `segmentation.available_models`. Each is built, fed real
image+mask dataloaders, and trained - with the before/after weight check above.

In [ ]:
seg_plans = PM._segmentation_plans(pm, EPOCHS)
print('Segmentation models to train:', [p.architecture for p in seg_plans])
print()
seg_results = {}
for plan in seg_plans:
    seg_results[plan.name] = train_one_model(plan, 'SEGMENTATION')

## 10 - Stage 5 - Prediction models (per tissue)

One plan per tissue (`hb_<tissue>`), routed to its configured backbone. This is where
the Hb regression actually happens - watch `val_mae` fall across epochs and the weight
signature change.

In [ ]:
pred_plans = PM._prediction_plans(pm, EPOCHS)
print('Prediction models to train:', [(p.name, p.architecture) for p in pred_plans])
print()
pred_results = {}
for plan in pred_plans:
    pred_results[plan.name] = train_one_model(plan, 'PREDICTION')

print()
print('Registry after training:')
print(json.dumps(pm.registry.report(), indent=2, default=str)[:1500])

## 11 - Stage 6 - Agents: every adaptive decision, out loud

The perception -> decision -> clinical agents run in order through the
`WorkflowController`, each writing structured outputs the next one reads. Below we list
every enabled agent and run the workflow on a demo per-tissue context so you can see
exactly what each agent decides and where it plugs into the flow.

In [ ]:
print('Enabled agents (execution order):')
for i, name in enumerate(pm.agents.enabled_agents(), 1):
    print(f'  {i}. {name}')
print()

demo_context = {
    'tissues': {t: {} for t in pm.dataset.dataset_config.tissues},
    'tissue_estimates': {t: 12.5 for t in pm.dataset.dataset_config.tissues},
    'tissue_confidences': {t: 0.8 for t in pm.dataset.dataset_config.tissues},
}
result = pm.agents.run_workflow(demo_context)

print('Per-agent decisions:')
for d in result.decisions:
    agent_name = getattr(d, 'agent', None) or getattr(d, 'name', 'agent')
    outs = {k: v for k, v in d.outputs.items() if k not in ('sample_weights',)}
    print(f'  - {agent_name}: {d.reason}')
    if outs:
        print('       outputs:', json.dumps(outs, default=str)[:200])
print()
print('Fused final_hb     :', result.final_hb)
print('Confidence         :', result.confidence)
print('Recommendation     :', result.recommendation)

## 12 - Stage 7 - Evaluation + **prediction provenance** (the MAE explainer)

This is the cell that explains a bad MAE. Before computing metrics we check, per tissue:
did evaluation find a **trained checkpoint**, and what does the loaded model actually
**predict on a real test batch**? A constant near 0 (while true Hb ~ 12) is exactly how
you get MAE ~ 12 - it means the model is untrained/collapsed or the checkpoint didn't load.

In [ ]:
from adaptivehb.core.types import ModelCategory

print('Registered PREDICTION models:', [r.name for r in pm.registry.find(ModelCategory.PREDICTION)])
print()
print('Per-tissue prediction provenance:')
test_samples = pm.dataset.samples('test') or pm.dataset.samples()
for tissue in pm.dataset.dataset_config.tissues:
    name = f'hb_{tissue}'
    has_ckpt = pm.checkpoints.exists(name, tag='best') or pm.checkpoints.exists(name, tag='latest')
    model = pm.prediction.load_trained(name, pm.checkpoints, tissue=tissue)
    tsamp = [s for s in test_samples if s.tissue == tissue][:8]
    try:
        preds = pm.prediction.predict_samples(model, tsamp) if tsamp else []
    except Exception as e:
        preds = f'ERROR: {e}'
    kind = model_kind(model)
    pv = ([round(p, 3) for p in preds] if isinstance(preds, list) else preds)
    print(f'  {tissue:7s}: checkpoint={"YES" if has_ckpt else "NO "} | model={kind} | preds(sample)={pv}')
    if isinstance(preds, list) and preds and max(abs(p) for p in preds) < 1.0:
        print('           WARNING: predictions ~ 0 while true Hb ~ 12 -> this is the MAE~12 cause.')
print()

eval_out = pipeline.evaluate()
m = eval_out['metrics']
print('Evaluation metrics (', eval_out['num_samples'], 'test patients ):')
for k in ('mae','rmse','r2','pearson','spearman','mean_bias'):
    if k in m:
        print(f'   {k:>10}: {m[k]:.4f}')

## 13 - Verdict - is it training, and are the models used?

A one-screen recap tying every stage together.

In [ ]:
def yn(b): return 'YES' if b else 'NO'

print('=' * 68)
print('ENVIRONMENT')
print('  PyTorch present            :', yn(TORCH_OK))
print('  Real prediction backbones  :', yn(REAL_PRED), '->', pred_avail)
print('  Real segmentation backbones:', yn(REAL_SEG), '->', seg_avail)
print('DATA')
print('  Patients / images          :', report.num_patients, '/', report.num_images)
print('  Split (patients)           :', {k: len(v) for k, v in split_ids.items()})
print('  Preprocessing agent ran    :', yn(agent is not None))
print('TRAINING')
print('  Segmentation models trained:', list(seg_results.keys()))
print('  Prediction models trained  :', list(pred_results.keys()))
print('  Models registered          :', [r.name for r in pm.registry.find(ModelCategory.PREDICTION)])
print('AGENTS')
print('  Agents executed            :', pm.agents.enabled_agents())
print('EVALUATION')
print('  Test MAE                   :', round(m.get('mae', float('nan')), 4))
print('=' * 68)
if not REAL_PRED:
    print('VERDICT: models are NOT learning - reference fallback. Install the ML extra and re-run.')
elif m.get('mae', 99) > 5:
    print('VERDICT: real backbones ARE training, but eval MAE is high. Check the provenance')
    print('         cell above - likely untrained/collapsed head or checkpoint not loaded.')
else:
    print('VERDICT: training + evaluation look healthy.')

pipeline.shutdown()